How useful is the endpoint score?

In [ ]:
### import the MT file and take the overlapping chemials within the MT groups ###
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
MT_PATH = r"Q:\Messdaten\EXPECTmine_data\02_MLinvitroTox_PosNeg\mlinvitrotox\data\results\models\final_models_retrained\selected_aeids.csv"
df_mt = pd.read_csv(MT_PATH)
aeids = df_mt["aeid"].astype(str)

In [ ]:
### try to find the code to get proba and use this instead

import os
import glob

probability_dir = r"Q:\Messdaten\EXPECTmine_data\02_MLinvitroTox_PosNeg\proba_final_models_retrained"


dfs = []
for f in glob.glob(os.path.join(probability_dir, "*_test_pred.csv")):
    aeid = os.path.basename(f).split("_")[0]
    df = pd.read_csv(f)
    df["aeid"] = aeid
    dfs.append(df)

analysis_df = pd.concat(dfs, ignore_index=True)

In [ ]:
# get balanced accuracy for each aeid from the metrics.csv file
class_dir = Path(r"Q:\Messdaten\EXPECTmine_data\02_MLinvitroTox_PosNeg\mlinvitrotox\data\results\models\final_models_retrained\class")

balanced_accuracy = []

for aeid in aeids:
    metrics_path = (class_dir/ str(aeid)/ "FS_RandomForest"/ "mb_val_structure"/ "metrics.csv")

    metrics_df = pd.read_csv(metrics_path)

    balanced_accuracy.append({
        'aeid': str(aeid),
        'balanced_accuracy': metrics_df.iloc[0]['balanced_accuracy']
    })

balanced_accuracy_df = pd.DataFrame(balanced_accuracy)

In [ ]:
analysis_df["aeid"] = analysis_df["aeid"].astype(str)
df_mt["aeid"] = df_mt["aeid"].astype(str)

df_merged = pd.merge(
    analysis_df,
    df_mt,
    on="aeid",
    how="left"
)

df_merged = pd.merge(
    df_merged,
    balanced_accuracy_df,
    on="aeid",
    how="left"
)

In [ ]:

def calculate_endpoint_score(predictions_df):
    # remove not annotated mechanistic targets
    filtered_predictions_df = predictions_df[predictions_df["MechanisticTarget"] != "not_annotated"].copy()

    # group and calculate endpoint score
    cols_gb = ["chem_id", "signal_direction", "MechanisticTarget"]
    filtered_predictions_df["endpoint_score"] = (
        filtered_predictions_df.groupby(cols_gb)["y_pred"]
        .transform("mean")
    )
    predictions_df = predictions_df.merge(
        filtered_predictions_df[cols_gb + ["endpoint_score"]],
        on=cols_gb,
        how="left"
    )
    predictions_df["endpoint_score"] = predictions_df["endpoint_score"].fillna(np.nan)

    # calculate number of endpoints per mechanistic target
    predictions_df["endpoints_count"] = (
        predictions_df.groupby(cols_gb[1:])["aeid"]
        .transform("nunique")
    )
    # weighted endpoint score based on model balanced accuracy
    filtered_predictions_df["weighted_prob"] = (filtered_predictions_df["y_proba"]* filtered_predictions_df["balanced_accuracy"])

    weighted_endpoint_score = (
        filtered_predictions_df
        .groupby(cols_gb)
        .agg(
            weighted_prob_sum=("weighted_prob", "sum"),
            ba_sum=("balanced_accuracy", "sum")
        )
        .reset_index()
    )

    weighted_endpoint_score["weighted_endpoint_score"] = (weighted_endpoint_score["weighted_prob_sum"]/ weighted_endpoint_score["ba_sum"])   

    weighted_endpoint_score = weighted_endpoint_score[cols_gb + ["weighted_endpoint_score"]]

    predictions_df = predictions_df.merge(
        weighted_endpoint_score,
        on=cols_gb,
        how="left"
    )

    predictions_df = predictions_df.drop_duplicates()

    return predictions_df

predictions_df = calculate_endpoint_score(df_merged)

In [15]:
df_filter = predictions_df[predictions_df['MechanisticTarget'] != "not_annotated"]
df_filter.head()

,chem_id,y_true,y_pred,y_proba,aeid,assay_component_endpoint_name,MechanisticTarget,signal_direction,balanced_accuracy,endpoint_score,endpoints_count,weighted_endpoint_score
477,DTXSID0020020,0,0,0.470046,100,ATG_Pax6_CIS,Cell Cycle,gain,0.65654,0.00,4,0.251982
481,DTXSID0020022,0,1,0.501406,100,ATG_Pax6_CIS,Cell Cycle,gain,0.65654,0.75,4,0.448462
485,DTXSID0020311,0,1,0.505181,100,ATG_Pax6_CIS,Cell Cycle,gain,0.65654,0.75,4,0.465289
489,DTXSID0020446,1,1,0.505181,100,ATG_Pax6_CIS,Cell Cycle,gain,0.65654,0.75,4,0.470258
493,DTXSID0020523,0,0,0.470046,100,ATG_Pax6_CIS,Cell Cycle,gain,0.65654,0.00,4,0.338811


Do the analysis only for compounds that the model has predicted as active, y_pred = 1, as these are the ones I will be focusing on

In [17]:
thresholds = np.arange(0, 1.01, 0.05)

rows = []

for thr in thresholds:
    pred = (df_filter["weighted_endpoint_score"] >= thr).astype(int)

    tmp = df_filter.copy()
    tmp["pred_thr"] = pred

    for (mechanistic, signal), g in tmp.groupby(["MechanisticTarget", "signal_direction"]):

        tp = ((g.y_true == 1) & (g.y_pred == 1)).sum()
        fp = ((g.y_true == 0) & (g.y_pred == 1)).sum()
        tp_thr = ((g.y_true == 1) & (g.y_pred == 1) & (g.pred_thr == 1)).sum()
        fp_thr = ((g.y_true == 0) & (g.y_pred == 1) & (g.pred_thr == 1)).sum()

        rows.append({
            "MechanisticTarget": mechanistic,
            "signal_direction": signal,
            "variable": "weighted_endpoint_score",
            "threshold": thr,
            "TP": tp,
            "FP": fp,
            "TP_thr": tp_thr,
            "FP_thr": fp_thr
        })

roc_df_weighted_endpt = pd.DataFrame(rows)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages("TP_FP_vs_threshold_weighted_endpt_score_complete.pdf") as pdf:

    groups = (
        roc_df_weighted_endpt[
            ["MechanisticTarget", "signal_direction"]
        ]
        .drop_duplicates()
    )

    for _, grp in groups.iterrows():

        mechanistic = grp["MechanisticTarget"]
        signal = grp["signal_direction"]

        df_plot = (
            roc_df_weighted_endpt[
                (roc_df_weighted_endpt["MechanisticTarget"] == mechanistic) &
                (roc_df_weighted_endpt["signal_direction"] == signal)
            ]
            .sort_values("threshold")
        )

        fig, ax = plt.subplots(figsize=(8,5))

        ax.plot(
            df_plot["threshold"],
            df_plot["TP_thr"],
            marker="o",
            linewidth=2,
            label="True Positives"
        )

        ax.plot(
            df_plot["threshold"],
            df_plot["FP_thr"],
            marker="s",
            linewidth=2,
            label="False Positives"
        )

        ax.set_xlabel("Threshold")
        ax.set_ylabel("Count")
        ax.set_title(f"{mechanistic} ({signal})")
        ax.grid(True)
        ax.legend()

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)